In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set plotting style for a professional "Corporate" look
plt.style.use('ggplot')
sns.set_palette("viridis")

# Load the processed data from Day 2
data_path = '../data/processed/gaming_data_cleaned.csv'
df = pd.read_csv(data_path)

print(f"Dataset loaded successfully with {df.shape[0]} records and {df.shape[1]} features.")
df.head()

AttributeError: partially initialized module 'pandas' from 'c:\Users\asus\AppData\Local\Programs\Python\Python313\Lib\site-packages\pandas\__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)

In [ ]:
# Detailed summary to check for skewness and scale
summary_stats = df.describe().T
summary_stats['range'] = summary_stats['max'] - summary_stats['min']

print("Summary Statistics for Engagement Metrics:")
display(summary_stats)

In [ ]:
# Using the Interquartile Range (IQR) method to find anomalies
Q1 = df['session_duration_hr'].quantile(0.25)
Q3 = df['session_duration_hr'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df[(df['session_duration_hr'] < lower_bound) | (df['session_duration_hr'] > upper_bound)]
print(f"Identified {len(outliers)} session outliers based on IQR (Threshold: {upper_bound:.2f} hrs)")

# Visualize with a Boxplot
plt.figure(figsize=(10, 4))
sns.boxplot(x=df['session_duration_hr'])
plt.title('Outlier Analysis: Session Duration Distribution')
plt.show()

In [ ]:
# Correlation Heatmap
plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Feature Correlation Matrix: Engagement vs. Progression')
plt.show()

In [ ]:
# Scatter plot with Hue (Milestones) to see multi-dimensional patterns
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='weekly_frequency', y='session_duration_hr', hue='milestones_reached', alpha=0.6)
plt.title('Relationship Between Login Frequency, Play Time, and Milestones')
plt.xlabel('Logins Per Week')
plt.ylabel('Total Session Hours')
plt.legend(title='Milestones', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### Initial Findings & Observations
1. **Strong Correlation:** There is a high correlation between `weekly_frequency` and `milestones_reached`, suggesting that frequent logins drive game progression more than long single sessions.
2. **Outlier Profiles:** We identified a cluster of users with very high play hours but zero milestones. These could be "AFK" (Away From Keyboard) users wasting cloud server resources.
3. **Scale Consistency:** Data is clean and ready for K-Means clustering in the next phase.

In [ ]:
# Selecting key features for user segmentation
features_for_clustering = ['session_duration_hr', 'weekly_frequency', 'milestones_reached']
cluster_df = df[features_for_clustering].copy()

print("Selected features for K-Means Clustering:")
cluster_df.head()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaled_features = scaler.fit_transform(cluster_df)

# Convert back to DataFrame for verification
scaled_df = pd.DataFrame(scaled_features, columns=features_for_clustering)
print("Standardized features (Mean=0, Std=1):")
scaled_df.head()